In [ ]:
import subprocess
import sys
try:
    from catboost import CatBoostClassifier, Pool
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "catboost", "-q"])
    from catboost import CatBoostClassifier, Pool

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_curve, auc, precision_recall_curve
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('Augmented Dataset of Nigeria Crash.csv')

drop_cols = [col for col in df.columns if col.startswith('State_') or col.startswith('Region_')]
drop_cols += ['State_Encoded', 'Region_Encoded', 'Quarter_Num', 'Year', 'Quarter_Date', 'High_Casualty']
df = df.drop(columns=drop_cols, errors='ignore')

categorical_features = ['Quarter', 'State', 'Region']
for col in categorical_features:
    df[col] = df[col].astype('category')

X = df.drop('Severe_Crash', axis=1)
y = df['Severe_Crash'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

cat_feature_indices = [X.columns.get_loc(col) for col in categorical_features if col in X.columns]

train_pool = Pool(X_train, y_train, cat_features=cat_feature_indices)
test_pool = Pool(X_test, y_test, cat_features=cat_feature_indices)

param_grid = {
    'iterations': [100, 200, 300],
    'depth': [4, 6, 8, 10],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'l2_leaf_reg': [1, 3, 5, 7]
}

model = CatBoostClassifier(verbose=0, random_seed=42, auto_class_weights='Balanced')
randomized_search_result = model.randomized_search(param_grid, train_pool, n_iter=10, cv=3, verbose=False, plot=False)

best_model = CatBoostClassifier(verbose=0, random_seed=42, auto_class_weights='Balanced', **randomized_search_result['params'])
best_model.fit(train_pool)

train_pred = best_model.predict(X_train)
test_pred = best_model.predict(X_test)
train_score = accuracy_score(y_train, train_pred)
test_score = accuracy_score(y_test, test_pred)
gap = train_score - test_score

print(f"Train Accuracy: {train_score:.4f}")
print(f"Test Accuracy: {test_score:.4f}")
print(f"Train-Test Gap: {gap:.4f}")
print("\nClassification Report (Test):\n", classification_report(y_test, test_pred))

conf_mat = confusion_matrix(y_test, test_pred)
plt.figure(figsize=(6,5))
sns.heatmap(conf_mat, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show ()
plt.savefig('confusion_matrix.png')
plt.close()

y_test_proba = best_model.predict_proba(X_test)[:,1]
fpr, tpr, _ = roc_curve(y_test, y_test_proba)
roc_auc = auc(fpr, tpr)
plt.figure(figsize=(6,5))
plt.plot(fpr, tpr, label=f'ROC curve (AUC = {roc_auc:.2f})')
plt.plot([0,1],[0,1],'k--')
plt.xlim([0,1])
plt.ylim([0,1])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend(loc='lower right')
plt.show ()
plt.savefig('roc_curve.png')
plt.close()

precision, recall, _ = precision_recall_curve(y_test, y_test_proba)
plt.figure(figsize=(6,5))
plt.plot(recall, precision, label='Precision-Recall curve')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend()
plt.show ()
plt.savefig('precision_recall_curve.png')
plt.close()

feature_importance = best_model.get_feature_importance()
feature_names = X.columns
sorted_idx = np.argsort(feature_importance)[::-1]
plt.figure(figsize=(10,8))
plt.barh(range(len(sorted_idx[:20])), feature_importance[sorted_idx[:20]], align='center')
plt.yticks(range(len(sorted_idx[:20])), [feature_names[i] for i in sorted_idx[:20]])
plt.gca().invert_yaxis()
plt.xlabel('Feature Importance')
plt.title('Top 20 Feature Importances')
plt.savefig('feature_importance.png')
plt.close()

best_model.fit(train_pool, eval_set=test_pool, verbose=False, plot=True)
plt.show ()
plt.savefig('learning_curve.png')
plt.close()

results_df = pd.DataFrame({
    'Metric': ['Train Accuracy', 'Test Accuracy', 'Train-Test Gap'],
    'Value': [train_score, test_score, gap]
})
results_df.to_csv('model_scores.csv', index=False)
print("All graphs and results saved successfully.")

print("\n")
Train_Score = accuracy_score(y_train, train_pred)
Test_Score = accuracy_score(y_test, test_pred)
print(f"Train Accuracy: {Train_Score:.4f}")
print(f"Test Accuracy: {Test_Score:.4f}")